# 📝 01 - Configuração inicial do banco de dados (MySQL)

Este notebook cria (se ainda não existirem) as tabelas:

- `pessoas` → cadastro das pessoas acolhidas  
- `usuarios` → usuários que podem acessar o sistema (login)

As configurações de conexão são lidas do arquivo `mysql_config.json`
gerado pelo notebook **`00_mysql_config.ipynb`**.

> Passos:
> 1. Execute o `00_mysql_config.ipynb` (ele grava o `mysql_config.json`).
> 2. Depois, execute a célula de código abaixo para criar/verificar as tabelas.


In [5]:
import json
from pathlib import Path
import mysql.connector

# Caminho do arquivo de configuração gerado no 00_mysql_config.ipynb
CONFIG_PATH = Path("mysql_config.json")


def carregar_config_mysql() -> dict:
    """
    Carrega as configurações de conexão do arquivo mysql_config.json.
    Esse arquivo é criado pelo notebook 00_mysql_config.ipynb.
    """
    if not CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Arquivo de configuração {CONFIG_PATH} não encontrado.\n"
            "Execute antes o notebook 00_mysql_config.ipynb para gerar o mysql_config.json."
        )
    return json.loads(CONFIG_PATH.read_text(encoding="utf-8"))


def get_connection(cfg: dict | None = None):
    """
    Abre uma conexão com o MySQL usando as configurações do arquivo.
    Assume que o database (bd_unl) já existe na hospedagem.
    """
    if cfg is None:
        cfg = carregar_config_mysql()

    conn = mysql.connector.connect(
        host=cfg["host"],
        port=cfg["port"],
        user=cfg["user"],
        password=cfg["password"],
        database=cfg["database"],  # aqui já conecta direto no bd_unl
    )
    return conn


def init_db():
    """
    Cria/atualiza as tabelas `pessoas` e `usuarios` dentro do database
    configurado em mysql_config.json.
    Não tenta criar o database, pois em hospedagens compartilhadas isso
    normalmente não é permitido.
    """
    cfg = carregar_config_mysql()
    conn = get_connection(cfg)
    cur = conn.cursor()

    # ---- Tabela de pessoas acolhidas ----
    create_pessoas_sql = """
    CREATE TABLE IF NOT EXISTS pessoas (
        id INT AUTO_INCREMENT PRIMARY KEY,
        nome VARCHAR(200) NOT NULL,
        apelido VARCHAR(100),
        data_nascimento DATE,
        documento_principal VARCHAR(100),
        tem_documentos TINYINT(1) DEFAULT 0,   -- 0 = não, 1 = sim
        telefone VARCHAR(50),
        contato_emergencia VARCHAR(200),
        cidade_origem VARCHAR(150),
        situacao_rua_desde VARCHAR(255),
        saude_resumo TEXT,
        dependencias_quimicas TEXT,
        observacoes TEXT,
        status VARCHAR(20) DEFAULT 'ativo',    -- ativo / inativo / encaminhado
        data_cadastro DATETIME DEFAULT CURRENT_TIMESTAMP
    )
    """

    # ---- Tabela de usuários do sistema (login) ----
    create_usuarios_sql = """
    CREATE TABLE IF NOT EXISTS usuarios (
        id INT AUTO_INCREMENT PRIMARY KEY,
        nome VARCHAR(200) NOT NULL,
        email VARCHAR(200) NOT NULL UNIQUE,
        senha_hash VARCHAR(255) NOT NULL,
        perfil VARCHAR(50) DEFAULT 'colaborador',  -- admin / colaborador
        ativo TINYINT(1) DEFAULT 1,
        criado_em DATETIME DEFAULT CURRENT_TIMESTAMP
    )
    """

    # Executa os CREATE TABLE
    cur.execute(create_pessoas_sql)
    cur.execute(create_usuarios_sql)
    conn.commit()

    print("[OK] Tabela 'pessoas' verificada/criada.")
    print("[OK] Tabela 'usuarios' verificada/criada.")

    cur.close()
    conn.close()


# Basta executar esta célula para criar/verificar as tabelas.
init_db()

[OK] Tabela 'pessoas' verificada/criada.
[OK] Tabela 'usuarios' verificada/criada.
